# RAG Pipeline

In [1]:
import importlib
import evaluation.eval
importlib.reload(evaluation.eval)
from implementation.ingest import load_hotpotqa_corpus_jsonl, create_sentence_chunks, create_embeddings
from implementation.answer import fetch_context, answer_question
from evaluation.eval import evaluate_retrievals, evaluate_answers, run_cli_evaluation


c:\Users\igor.banovic\projects\llm_engineering\week9\implementation\answer.py:53: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
# load HotpotQA corpus from JSONL file
rows = load_hotpotqa_corpus_jsonl()
print(len(rows))
print(rows[0])

Loaded 11,742 corpus title-docs from hotpotqa_unique_pool1200_corpus_titles.jsonl
11742
{'id': '"The Spaghetti Incident?"', 'title': '"The Spaghetti Incident?"', 'sentences': ['"The Spaghetti Incident?"', "is the fifth studio album by the American hard rock band Guns N' Roses.", 'The album is composed of covers of older punk rock and hard rock songs. "', '"The Spaghetti Incident?""', 'is the only studio album to feature rhythm guitarist Gilby Clarke, who replaced original Guns N\' Roses member Izzy Stradlin during the band\'s "Use Your Illusion" tour in 1991, as well as the last album to feature guitarist Slash, bassist Duff McKagan and drummer Matt Sorum.', "It is also the only Guns N' Roses album not to be accompanied by a supporting tour."]}


In [3]:
# Create sentence chunks from the loaded corpus
chunks = create_sentence_chunks(rows)
print(len(chunks))
print(chunks[0])

Created 13,361 sentence-chunks (SENTS_PER_CHUNK=6, SENTS_OVERLAP=1)
13361
page_content='"The Spaghetti Incident?" is the fifth studio album by the American hard rock band Guns N' Roses. The album is composed of covers of older punk rock and hard rock songs. " "The Spaghetti Incident?"" is the only studio album to feature rhythm guitarist Gilby Clarke, who replaced original Guns N' Roses member Izzy Stradlin during the band's "Use Your Illusion" tour in 1991, as well as the last album to feature guitarist Slash, bassist Duff McKagan and drummer Matt Sorum. It is also the only Guns N' Roses album not to be accompanied by a supporting tour.' metadata={'id': '"The Spaghetti Incident?"', 'title': '"The Spaghetti Incident?"', 'chunk_index': 0, 'sent_start': 0, 'sent_end': 6}


In [4]:
# Create vector database by creating embeddings for the chunks
create_embeddings(chunks)

c:\Users\igor.banovic\projects\llm_engineering\week9\implementation\ingest.py:30: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(model_name=model_name)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success: 13,361 vectors created with 384 dimensions.
DB_NAME = C:\Users\igor.banovic\projects\llm_engineering\week9\vector_db


In [ ]:
fetch_context("Which mine was established earlier, Murray Brook Mine or Temagami-Lorrain Mine?")

In [ ]:
answer_question("Which mine was established earlier, Murray Brook Mine or Temagami-Lorrain Mine?")

In [ ]:
# Create a reproducible HotpotQA subset and store it as JSONL.
%run implementation/make_hotpotqa_subsets.py

In [2]:
# Build / refresh the local vector database from the knowledge base
%run implementation/ingest.py

Corpus path: C:\Users\igor.banovic\projects\llm_engineering\week9\data\hotpotqa_unique_pool1200_corpus_titles.jsonl
EMBEDDING_MODEL_NAME = all-MiniLM-L12-v2
Loaded 11,742 corpus title-docs from hotpotqa_unique_pool1200_corpus_titles.jsonl
Created 13,361 sentence-chunks (SENTS_PER_CHUNK=6, SENTS_OVERLAP=1)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Success: 13,361 vectors created with 384 dimensions.
DB_NAME = C:\Users\igor.banovic\projects\llm_engineering\week9\vector_db
EMBEDDING_MODEL_NAME = all-MiniLM-L12-v2
Ingestion complete


In [15]:
run_cli_evaluation(row_number=9, k=10, structured_context=True, prompt_version="v2")

Example #9 (id=5a83478a55429954d2e2ec7d)
Question: The star of the 2010 comedy film High School won an Academy Award for Best Actor for his appearance in what movie?
Type/Level: bridge / medium
Reference Answer: "The Pianist"
Supporting facts (2): [('High School (2010 film)', 0), ('Adrien Brody', 1)]
Retrieval recall@10: 0.5000 (1/2)
Retrieval MRR@10: 0.5000
Retrieval nDCG@10: 1.0000
IGOR: prompt_version: v2
### answer_question():
 question: The star of the 2010 comedy film High School won an Academy Award for Best Actor for his appearance in what movie?,
 history: [],
 k: 10,
 structured_context: True,
 prompt_version: v2,
 system_prompt: 
You are a helpful assistant answering questions based on Wikipedia articles.

Answer the question using ONLY the facts stated in the context below.
- If the context doesn't contain enough information, say "I don't have enough information in the context to answer that."
- Do not speculate or make inferences beyond what's explicitly given.
- Be direct

In [ ]:
# Build / refresh the local vector database from the knowledge base
# --first=1 --structured-context --k=10
#--prompt-version v1 | v2 # v1 - begin, v2 - short, only from context 
%run evaluator.py --first=0 --prompt-version v2 --k=2
#%run evaluator.py --first=0 --k=10
#%run evaluator.py --first=99


C:\Users\igor.banovic\projects\llm_engineering\week9\evaluator.py:944: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="RAG Evaluation Dashboard", theme=theme) as app:


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
